# 🦀 Day 3: Feature 3 — Publish/Subscribe Messaging
---

Today we add Pub/Sub to RustKV.

- **Pub/Sub pattern**: Publishers send messages; subscribers receive on channels
- **Commands**: SUBSCRIBE channel, UNSUBSCRIBE, PUBLISH channel message
- **Implementation**: `tokio::sync::broadcast` for message distribution
- **Pattern subscriptions**: Glob matching (e.g., `news:*`)
- **Exercise**: Add pattern-based subscriptions (PSUBSCRIBE)

## Pub/Sub with tokio::sync::broadcast

Each channel has a broadcast sender; subscribers hold receivers. Messages sent to the channel are received by all subscribers.

In [ ]:
:dep tokio = { version = "1", features = ["sync"] }

use std::collections::HashMap;
use std::sync::Arc;
use tokio::sync::{broadcast, RwLock};

type ChannelName = String;
type Message = (ChannelName, String);

pub struct PubSubManager {
    channels: Arc<RwLock<HashMap<ChannelName, broadcast::Sender<Message>>>>,
}

impl PubSubManager {
    pub fn new() -> Self {
        Self { channels: Arc::new(RwLock::new(HashMap::new())) }
    }

    pub async fn subscribe(&self, channel: &str) -> broadcast::Receiver<Message> {
        let mut ch = self.channels.write().await;
        let tx = ch.entry(channel.to_string())
            .or_insert_with(|| broadcast::channel(100).0)
            .clone();
        tx.subscribe()
    }

    pub async fn publish(&self, channel: &str, message: &str) -> usize {
        let ch = self.channels.read().await;
        if let Some(tx) = ch.get(channel) {
            let _ = tx.send((channel.to_string(), message.to_string()));
            tx.receiver_count()
        } else {
            0
        }
    }

    pub async fn unsubscribe(&self, channel: &str) {
        let mut ch = self.channels.write().await;
        if let Some(tx) = ch.get(channel) {
            if tx.receiver_count() == 0 {
                ch.remove(channel);
            }
        }
    }
}

println!("PubSubManager defined with subscribe, publish, unsubscribe.");

In [ ]:
// Subscriber notification flow — how a client receives messages

let flow = r#"
// In handle_client, when SUBSCRIBE is received:
let mut rx = pubsub.subscribe(&channel).await;
// Switch client into 'subscriber mode' — no longer executes normal commands
loop {
    tokio::select! {
        msg = rx.recv() => {
            match msg {
                Ok((ch, payload)) => write_response(&mut stream, &format!("message {} {}", ch, payload)).await;
                Err(_) => break;
            }
        }
        // Also listen for UNSUBSCRIBE from same stream
        cmd = read_next_command(&mut stream) => { if cmd == "UNSUBSCRIBE" { break; } }
    }
}
"#;

println!("Subscriber flow:");
println!("{}", flow);

In [ ]:
// Integration with TCP server — command routing

fn route_pubsub(cmd: &[&str], _pubsub: &PubSubManager) -> &'static str {
    match cmd {
        ["SUBSCRIBE", _channel] => {
            // Return receiver to caller; caller spawns task to forward messages
            "+OK\r\n"
        }
        ["PUBLISH", _channel, _msg] => {
            // let n = pubsub.publish(channel, msg).await;
            // format!":{}", n)
            "+1\r\n"
        }
        ["UNSUBSCRIBE", _channel] => "+OK\r\n",
        _ => "-ERR unknown command\r\n",
    }
}

println!("PubSub commands: SUBSCRIBE, PUBLISH, UNSUBSCRIBE.");

## 📝 Exercise: Pattern-based subscriptions (PSUBSCRIBE)

Add `PSUBSCRIBE pattern` — subscribe to all channels matching a glob pattern (e.g., `news:*`). When a message is published to any matching channel, forward it to the subscriber. Use a pattern matcher (e.g., `glob` crate or manual `*` matching).

In [ ]:
// YOUR CODE HERE
// Add PSUBSCRIBE: store (pattern, receiver) per client
// On PUBLISH, check all patterns; if channel matches, send to that subscriber
// Pattern match: "news:*" matches "news:sports", "news:tech"

// fn matches_pattern(pattern: &str, channel: &str) -> bool {
//     // Convert glob to regex or iterate segments
// }

println!("Implement PSUBSCRIBE with glob pattern matching.");

## 🎯 Key Takeaways

1. Pub/Sub: publishers send to channels; subscribers receive all messages on subscribed channels
2. Use `tokio::sync::broadcast` — one sender, many receivers; each message goes to all
3. SUBSCRIBE puts client in subscriber mode; switch from command execution to message forwarding
4. Track channels per client; UNSUBSCRIBE drops the receiver
5. Pattern subscriptions (PSUBSCRIBE) require matching published channels against stored patterns

---
**Tomorrow:** Error handling & edge cases